In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
df_customers=spark.read.format("delta").load("/Volumes/workspace/bronze_ecommerce/customers/data/")

In [0]:
df_customers.printSchema()

In [0]:
df_customers.show()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df_silver_customers = (
    df_customers

    # Standardize data types first
    .withColumn("created_date", to_date(col("created_date"), "yyyy-MM-dd"))
    .withColumn("modified_date", to_date(col("modified_date"), "yyyy-MM-dd"))

    # Clean string columns
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("phone", coalesce(col("phone"), lit("UNKNOWN")))
    .withColumn("city", coalesce(trim(col("city")), lit("UNKNOWN")))
    .withColumn("state", coalesce(trim(col("state")), lit("UNKNOWN")))
    .withColumn("customer_segment", coalesce(trim(col("customer_segment")), lit("UNKNOWN")))

    # Email validation
    .withColumn(
        "email_valid",
        when(
            col("email").isNotNull() &
            col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"),
            lit(True)
        ).otherwise(lit(False))
    )

    # Latest record for each customer
    .withColumn(
        "row_number",
        row_number().over(
            Window
            .partitionBy("customer_id")
            .orderBy(desc("modified_date"))
        )
    )

    .filter(col("row_number") == 1)
    .drop("row_number", "_rescued_data")
)

In [0]:
df_silver_customers.printSchema()

In [0]:
df_products=spark.read.format("delta").load("/Volumes/workspace/bronze_ecommerce/products/data/")

In [0]:
df_products.printSchema()
df_products.show()

In [0]:
df_silver_products = (
    df_products

    .withColumn(
        "product_name",
        coalesce(
            trim(col("product_name")),
            lit("UNKNOWN")
        )
    )

    .withColumn(
        "category",
        coalesce(
            trim(col("category")),
            lit("UNKNOWN")
        )
    )

    .withColumn(
        "subcategory",
        coalesce(
            trim(col("subcategory")),
            lit("UNKNOWN")
        )
    )

    .withColumn("price", col("price").cast("double"))
    .withColumn(
        "price",
        when(
            col("price").isNull() | (col("price") < 0),
            lit(0.0)
        ).otherwise(col("price"))
    )

    .withColumn(
        "cost",
        when(
            col("cost").cast("double").isNull() |
            (col("cost").cast("double") < 0),
            lit(0.0)
        ).otherwise(
            col("cost").cast("double")
        )
    )

    .withColumn(
        "stock_quantity",
        when(
            col("stock_quantity").cast("int").isNull() |
            (col("stock_quantity").cast("int") < 0),
            lit(0)
        ).otherwise(
            col("stock_quantity").cast("int")
        )
    )

    .withColumn(
        "modified_date",
        to_date(
            col("modified_date"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "processed_date",
        current_timestamp()
    )
    .drop("_rescued_data")
)

In [0]:
df_silver_products.show()

In [0]:
df_payments=spark.read.format("delta").load("/Volumes/workspace/bronze_ecommerce/payments/data/")

In [0]:
df_payments.printSchema()

In [0]:
df_silver_payments=(
    df_payments
    .withColumn(
        "payment_amount",
        when(
            col("payment_amount").cast("double").isNull() |
            (col("payment_amount").cast("double") < 0),
            lit(0.0)
        ).otherwise(
            col("payment_amount").cast("double")))
    
    .withColumn(
        "payment_timestamp",
        to_timestamp(col("payment_timestamp"),"yyyy-MM-dd HH:mm:ss")
        )
                
    .withColumn(
        "payment_method",
        coalesce(
            trim(col("payment_method")),
            lit("UNKNOWN")
        )
    )
    .withColumn(
        "payment_status",
        coalesce(
            trim(col("payment_status")),
            lit("UNKNOWN")
        )
    )
    .withColumn(
        "processed_date",
        current_timestamp()
    )

    .withColumn("modified_date",to_date(col("modified_date"),"yyyy-MM-dd"))
    .drop("_rescued_data")
)

In [0]:
df_orders=spark.read.format("delta").load("/Volumes/workspace/bronze_ecommerce/orders/data/")
df_orders.printSchema()


In [0]:
df_silver_order=(
    df_orders
    .dropDuplicates()
    .withColumn(
        "order_amount",
        when(
            col("order_amount").cast("double").isNull() |
            (col("order_amount").cast("double") < 0),
            lit(0.0)
        ).otherwise(col("order_amount").cast("double"))
    )
    .withColumn(
        "quantity",
        when(
            col("quantity").cast("int").isNull() |
            (col("quantity").cast("int") < 0),
            lit(0)
        ).otherwise(col("quantity").cast("int"))
    )
    .withColumn(
        "order_timestamp",
        to_timestamp(col("order_timestamp"),"yyyy-MM-dd HH:mm:ss")
        )
    .withColumn(
        "order_status",
        coalesce(
            trim(col("order_status")),
            lit("UNKNOWN")
        )
    )
    .withColumn(
        "processed_date",
        current_timestamp()
    )
    .withColumn("modified_date",to_date(col("modified_date"),"yyyy-MM-dd"))
    .drop("_rescued_data")
    )
